# 서울시 골목상권 폐업위험 조기경보 — 데이터 준비와 탐색

> **핵심 질문** — 다음 분기에 점포 순감소(폐업 > 개업)로 전환될 상권 × 업종은 어디이며, 무엇이 위험을 만드는가

이 노트북은 **STEP 1 적재 → 2 결합 → 3 라벨 → 4 기초 EDA → 5 기준선·가설 EDA** 까지를 다룸.
예측 모델링(머신러닝)은 이 노트북의 산출물을 입력으로 하는 **다음 노트북**에서 수행함.

## 분석 단위와 기간

| 항목 | 값 |
|---|---|
| 분석 단위 | 상권 × 서비스업종 × 분기 |
| 업종 범위 | 요식업 10종 |
| 기간 | 2023Q1 ~ 2025Q4 (라벨 생성을 위해 2026Q1까지 적재) |
| 라벨 | `y(t) = 1 if 개업_점포_수(t+1) − 폐업_점포_수(t+1) < 0 else 0` |

## 데이터 출처

**서울시 우리마을가게 상권분석서비스** — 서울특별시 주관 / 서울신용보증재단 위탁운영
배포: 서울 열린데이터광장 (`data.seoul.go.kr`), 공공누리 1유형

원천은 추정매출 = 신한카드 결제액 ÷ 보정비율, 유동인구 = 서울시·KT 생활인구,
점포 = 국세청 사업자등록 기반임. 이 때문에 **"폐업"은 실제 폐점이 아니라 사업자등록 말소 시점**이며,
매출은 단일 카드사 기반 추정치임. 결론을 쓸 때 해석의 폭을 이 범위로 좁힘.

## 이 노트북에서 내린 판단

1. **점포 컬럼 대응** — 연도별 파일의 `유사_업종_점포_수` = 신 기준 `전체_점포_수`,
   `점포_수` = `일반_점포_수`. `전체 = 일반 + 프랜차이즈` 항등식으로 검증함.
2. **집객시설의 빈칸은 0임** — 원본에 `0` 표기가 한 건도 없음. `fillna(0)` 후 `역세권` 파생.
3. **인구는 총량 하나 + 구성 하나** — 유동인구가 상주·직장을 이미 포함하므로
   총량은 길단위인구, 구성은 `직장인구_비중` 하나로 요약함.
4. **2021~2022 제외** — 방역 조치가 개·폐업을 좌우한 구간.
5. **상권변화지표는 피처가 아니라 기준선** — 폐업 정보로 만들어져 라벨과 원천이 같음.

---
# STEP 1. 원본 적재와 스키마 통일

In [ ]:
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from scipy.stats import chi2_contingency, kruskal

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)

for cand in ["C:/Windows/Fonts/malgun.ttf", "/System/Library/Fonts/AppleSDGothicNeo.ttc"]:
    if Path(cand).exists():
        fm.fontManager.addfont(cand)
        plt.rcParams["font.family"] = fm.FontProperties(fname=cand).get_name()
        break
plt.rcParams["axes.unicode_minus"] = False

RAW = Path(os.environ.get(
    "SANGKWON_RAW",
    Path.home() / "OneDrive" / "바탕 화면" / "AI 데이터 인텔리전스" / "종합 과제" / "데이터"))
PROC = Path("data/processed"); PROC.mkdir(parents=True, exist_ok=True)
FIG  = Path("figures");        FIG.mkdir(parents=True, exist_ok=True)
ENC = "cp949"

Q_FEATURE = [20231, 20232, 20233, 20234, 20241, 20242, 20243, 20244,
             20251, 20252, 20253, 20254]
Q_ALL = Q_FEATURE + [20261]          # 20261은 20254의 라벨을 만들기 위해서만 보유함
MIN_STORES = 5                        # 표본 필터 기준
TARGET_TYPE = "골목상권"

FOOD = ["한식음식점", "중식음식점", "일식음식점", "양식음식점", "분식전문점",
        "패스트푸드점", "치킨전문점", "제과점", "커피-음료", "호프-간이주점"]

print("원본 폴더:", RAW, "| 존재:", RAW.exists())

In [ ]:
FILES = {
    "store_api": "서울시 상권분석서비스(점포-상권)_API.csv",
    "sales_api": "서울시 상권분석서비스(추정매출-상권)_API.csv",
    "area":      "서울시 상권분석서비스(영역-상권).csv",
    "flow":      "서울시 상권분석서비스(길단위인구-상권).csv",
    "work":      "서울시 상권분석서비스(직장인구-상권).csv",
    "resi":      "서울시 상권분석서비스(상주인구-상권).csv",
    "facility":  "서울시 상권분석서비스(집객시설-상권).csv",
    "change_ix": "서울시 상권분석서비스(상권변화지표-상권).csv",
}
FALLBACK_STORE = [
    "_연도별파일_보관/서울시_상권분석서비스(점포-상권)_2023년.csv",
    "_연도별파일_보관/서울시 상권분석서비스(점포-상권)_2024년.csv",
    "_연도별파일_보관/서울시 상권분석서비스(점포-상권).csv",
]
for k, v in FILES.items():
    print(f"  [{'OK' if (RAW/v).exists() else '없음':>3}] {k:11} {v}")

def read(name):
    p = RAW / name
    if not p.exists():
        raise FileNotFoundError(f"파일 없음: {p}")
    return pd.read_csv(p, encoding=ENC, low_memory=False)

## 1-1. 점포-상권

Open API 통합본은 21개 분기가 하나의 스키마로 들어 있어 연도별 파일을 이어 붙일 필요가 없음.
다만 **수집이 중간에 끊긴 파일을 그대로 쓰면 안 되므로**, 분기 수를 확인해 불완전하면
연도별 파일로 되돌아감.

| 2023 · 2024 | 2025~ |
|---|---|
| `유사_업종_점포_수` | `전체_점포_수` |
| `점포_수` | `일반_점포_수` |

컬럼명은 의미가 분명한 신 기준(전체 / 일반)으로 통일함.

In [ ]:
def unify_store(df):
    ren = {}
    if "유사_업종_점포_수" in df.columns: ren["유사_업종_점포_수"] = "전체_점포_수"
    if "점포_수" in df.columns:          ren["점포_수"] = "일반_점포_수"
    return df.rename(columns=ren)

store, src = None, None
if (RAW / FILES["store_api"]).exists():
    cand = unify_store(read(FILES["store_api"]))
    nq = cand["기준_년분기_코드"].nunique()
    if nq >= 21:
        store, src = cand, "Open API 통합본"
    else:
        print(f"  API 파일 분기 {nq}개 — 불완전으로 판단해 연도별 파일을 사용함")
if store is None:
    frames = [unify_store(pd.read_csv(RAW / r, encoding=ENC, low_memory=False))
              for r in FALLBACK_STORE]
    store, src = pd.concat(frames, ignore_index=True), "연도별 파일 3개 병합"

STORE_COLS = ["기준_년분기_코드", "상권_구분_코드", "상권_구분_코드_명", "상권_코드", "상권_코드_명",
              "서비스_업종_코드", "서비스_업종_코드_명",
              "전체_점포_수", "일반_점포_수", "프랜차이즈_점포_수",
              "개업_율", "개업_점포_수", "폐업_률", "폐업_점포_수"]
assert not set(STORE_COLS) - set(store.columns), f"컬럼 누락: {set(STORE_COLS)-set(store.columns)}"
store = store[STORE_COLS].copy()
print(f"출처: {src} · {len(store):,}행 · 분기 {store['기준_년분기_코드'].nunique()}개")

In [ ]:
# 컬럼 대응 검증 — 성립하지 않으면 두 컬럼을 반대로 매칭한 것이므로 중단함
chk = (store["전체_점포_수"] - (store["일반_점포_수"] + store["프랜차이즈_점포_수"])).abs()
ok = (chk < 1e-6).mean()
print(f"전체 = 일반 + 프랜차이즈 성립률 {ok:.4%}")
assert ok > 0.999, "컬럼 대응 오류 — 유사_업종_점포_수 ↔ 전체_점포_수 매칭을 확인할 것"
print("→ 대응 확인됨")

## 1-2. 나머지 테이블

`영역-상권`은 분기 개념이 없는 마스터임. 매출은 카드 결제가 잡히는 칸에만 존재해 점포보다 성김.

In [ ]:
SALES_COLS = ["기준_년분기_코드", "상권_코드", "서비스_업종_코드", "당월_매출_금액", "당월_매출_건수"]
sales    = read(FILES["sales_api"])[SALES_COLS].copy()
area     = read(FILES["area"])
flow     = read(FILES["flow"])
work     = read(FILES["work"])
resi     = read(FILES["resi"])
facility = read(FILES["facility"])
chg      = read(FILES["change_ix"])

for nm, df in [("점포", store), ("매출", sales), ("영역", area), ("길단위인구", flow),
               ("직장인구", work), ("상주인구", resi), ("집객시설", facility), ("상권변화지표", chg)]:
    q = df["기준_년분기_코드"].nunique() if "기준_년분기_코드" in df else 0
    print(f"  {nm:10} {len(df):>9,}행 · 상권 {df['상권_코드'].nunique():>5}개 · 분기 {q}")

### 집객시설의 빈칸은 결측이 아니라 0임

원본 파일 전체에 `0` 표기가 **한 건도 없고** 빈칸만 존재함.
`지하철_역_수`가 대부분 비어 있는 것은 "지하철역이 없는 상권"이라는 뜻임.
결측으로 판정해 변수를 버리면 **역세권이라는 강한 입지 변수를 통째로 잃음**.

In [ ]:
fac_cols = [c for c in facility.columns if c.endswith("_수")]
before = facility[fac_cols].isna().mean().mean()
facility[fac_cols] = facility[fac_cols].fillna(0)
facility["역세권"] = (facility["지하철_역_수"] > 0).astype(int)
print(f"집객시설 결측률 {before:.1%} → 0.0% (0으로 채움) · 역세권 상권 비율 {facility['역세권'].mean():.1%}")

### 상권 성격 — 직장인구 비중

유동인구(생활인구)는 상주·직장·방문이 이미 합쳐진 값임. 셋을 모두 넣으면 다중공선성이 커지므로
**총량은 길단위인구**, **구성은 직장인구 비중 하나**로 요약함.

같은 유동인구라도 오피스 상권은 평일 점심에 몰리고 주말에 비므로, 요식업 폐업 구조가 다름.
직장인구는 연 1회 갱신이라 시계열 변수가 아니라 **준(準)고정 상권 성격 변수**로 다룸.

In [ ]:
popmix = (work[["기준_년분기_코드", "상권_코드", "총_직장인구_수"]]
           .merge(resi[["기준_년분기_코드", "상권_코드", "총_상주인구_수"]],
                  on=["기준_년분기_코드", "상권_코드"], how="outer"))
den = popmix["총_직장인구_수"].fillna(0) + popmix["총_상주인구_수"].fillna(0)
popmix["직장인구_비중"] = np.where(den > 0, popmix["총_직장인구_수"].fillna(0) / den, np.nan)
print(popmix["직장인구_비중"].describe(percentiles=[.1, .5, .9]).round(3).to_string())

### 상권변화지표 — 기준선 전용

서울시가 이미 공식으로 쓰는 4분면 진단임(`운영_영업_개월_평균`·`폐업_영업_개월_평균`을 서울 평균과 비교).

**피처로 쓰지 않음.** 폐업 정보로 만들어져 우리 라벨과 원천이 같기 때문임.
모델에 넣으면 "폐업이 많아서 폐업한다"는 순환 설명이 되어 «무엇이 위험을 만드는가»가 무의미해짐.
대신 **모델이 넘어야 할 기준선**으로 STEP 5에서 사용함.

In [ ]:
IX_COLS = ["기준_년분기_코드", "상권_코드", "상권_변화_지표", "상권_변화_지표_명",
           "운영_영업_개월_평균", "폐업_영업_개월_평균"]
chg = chg[IX_COLS].copy()
chg["기준선_위험"] = (chg["상권_변화_지표"] == "HL").astype(int)   # HL = 상권축소
print(chg.groupby(["상권_변화_지표", "상권_변화_지표_명"]).size().rename("건수").to_string())
print(f"\n기준선(HL) 양성률 {chg['기준선_위험'].mean():.1%}")

## 1-3. 기간 필터

2021~2022년은 영업시간 제한 등 방역 조치가 개·폐업을 좌우한 구간임.
평상시의 위험 신호를 배우려는 모델에 넣으면 정책 충격을 상권 특성으로 잘못 학습하므로 제외함.
제외 판단의 근거로 연도별 개·폐업 수준을 남김.

In [ ]:
tmp = store[store["서비스_업종_코드_명"].isin(FOOD)].copy()
tmp["연도"] = tmp["기준_년분기_코드"] // 10
tmp["순증감"] = tmp["개업_점포_수"] - tmp["폐업_점포_수"]
display(tmp.groupby("연도").agg(행수=("폐업_점포_수", "size"),
                              평균개업=("개업_점포_수", "mean"),
                              평균폐업=("폐업_점포_수", "mean"),
                              순감소칸비율=("순증감", lambda s: (s < 0).mean())).round(3))

In [ ]:
def clip_q(df, name):
    if "기준_년분기_코드" not in df.columns: return df
    n0 = len(df); out = df[df["기준_년분기_코드"].isin(Q_ALL)].copy()
    print(f"  {name:12} {n0:>9,} → {len(out):>9,}")
    return out

print("기간 필터 2023Q1~2026Q1")
store    = clip_q(store, "점포");        sales    = clip_q(sales, "매출")
flow     = clip_q(flow, "길단위인구");     facility = clip_q(facility, "집객시설")
popmix   = clip_q(popmix, "popmix");    chg      = clip_q(chg, "상권변화지표")

# 키 유일성 — 결합 전에 반드시 확인함
for nm, df, k in [("점포", store, ["기준_년분기_코드","상권_코드","서비스_업종_코드"]),
                  ("매출", sales, ["기준_년분기_코드","상권_코드","서비스_업종_코드"]),
                  ("영역", area, ["상권_코드"]),
                  ("길단위인구", flow, ["기준_년분기_코드","상권_코드"]),
                  ("집객시설", facility, ["기준_년분기_코드","상권_코드"]),
                  ("popmix", popmix, ["기준_년분기_코드","상권_코드"]),
                  ("상권변화지표", chg, ["기준_년분기_코드","상권_코드"])]:
    assert df.duplicated(k).sum() == 0, f"{nm} 키 중복"
print("모든 테이블의 키가 유일함")

---
# STEP 2. 결합 — 요식업 패널 만들기

| 대상 | 결합 키 |
|---|---|
| 영역-상권 | `상권_코드` |
| 길단위인구 · 집객시설 · popmix · 상권변화지표 | `상권_코드` + `기준_년분기_코드` |
| 추정매출 | `상권_코드` + `기준_년분기_코드` + `서비스_업종_코드` |

좌결합 후 **행 수가 늘면 키 중복**이므로 즉시 중단함. 이 단계의 최대 위험임.

In [ ]:
n_all = len(store)
df = store[store["서비스_업종_코드_명"].isin(FOOD)].copy()
print(f"전체 업종 {n_all:,} → 요식업 {len(df):,} ({len(df)/n_all:.1%})")

# 밀도 분모는 요식업 필터 '전' 100개 업종 합으로 계산해야 정확함
tot_all = (store.groupby(["기준_년분기_코드", "상권_코드"])["전체_점포_수"]
           .sum().rename("상권_전체점포").reset_index())

def check(before, after, step):
    assert after == before, f"[{step}] 행 증식 {before:,} → {after:,}"
    print(f"  [{step:12}] {after:,} 유지")

n = len(df)
df = df.merge(area[["상권_코드", "상권_구분_코드_명", "자치구_코드_명", "행정동_코드_명",
                    "영역_면적", "엑스좌표_값", "와이좌표_값"]]
              .rename(columns={"상권_구분_코드_명": "상권유형", "자치구_코드_명": "자치구",
                               "행정동_코드_명": "행정동"}),
              on="상권_코드", how="left");                                check(n, len(df), "영역")
df = df.merge(flow[["기준_년분기_코드", "상권_코드", "총_유동인구_수"]],
              on=["기준_년분기_코드", "상권_코드"], how="left");            check(n, len(df), "길단위인구")
df = df.merge(facility[["기준_년분기_코드", "상권_코드", "집객시설_수", "지하철_역_수", "역세권"]],
              on=["기준_년분기_코드", "상권_코드"], how="left");            check(n, len(df), "집객시설")
df = df.merge(popmix[["기준_년분기_코드", "상권_코드", "직장인구_비중"]],
              on=["기준_년분기_코드", "상권_코드"], how="left");            check(n, len(df), "popmix")
df = df.merge(chg[["기준_년분기_코드", "상권_코드", "상권_변화_지표", "기준선_위험"]],
              on=["기준_년분기_코드", "상권_코드"], how="left");            check(n, len(df), "상권변화지표")
df = df.merge(sales, on=["기준_년분기_코드", "상권_코드", "서비스_업종_코드"],
              how="left");                                              check(n, len(df), "추정매출")
df = df.merge(tot_all, on=["기준_년분기_코드", "상권_코드"], how="left");    check(n, len(df), "상권집계")

## 2-1. 결측률

결측률이 높은 변수는 매칭·모델 공변량에서 빼야 함. 표본이 통째로 날아가기 때문임.

In [ ]:
cols = ["엑스좌표_값", "영역_면적", "총_유동인구_수", "집객시설_수", "지하철_역_수",
        "직장인구_비중", "상권_변화_지표", "당월_매출_금액", "상권_전체점포"]
miss = pd.DataFrame({"컬럼": cols, "결측률": [df[c].isna().mean() for c in cols]})
miss["판정"] = pd.cut(miss["결측률"], [-1, .05, .3, 1], labels=["OK", "주의", "공변량 제외 권장"])
miss["결측률"] = miss["결측률"].map("{:.1%}".format)
display(miss)
print("※ 매출 결측이 큰 이유는 결합 실수가 아니라, 카드 매출이 잡히지 않는 영세 칸에는 원본에 행이 없기 때문임.")
df.to_pickle(PROC / "02_merged.pkl")
print(f"[저장] 02_merged.pkl {df.shape}")

---
# STEP 3. 라벨 생성과 표본 필터

```
순증감(t) = 개업_점포_수(t) − 폐업_점포_수(t)
y(t)      = 1 if 순증감(t+1) < 0 else 0
```

두 가지를 지킴.
- `shift(-1)`은 반드시 **(상권, 업종) 그룹 내에서 분기 오름차순 정렬 후** 수행함
- 분기가 연속하지 않으면 라벨을 만들지 않음 (`20244 → 20251` 은 연속으로 처리함)

**필터 근거** — 점포 5개 미만 칸은 개·폐업이 거의 없어 라벨이 노이즈에 가까움.

In [ ]:
def quarter_index(code):
    """20244 → 8099, 20251 → 8100. 연도 경계에서도 diff가 1이 되도록 함"""
    return (code // 10) * 4 + (code % 10 - 1)

df = pd.read_pickle(PROC / "02_merged.pkl")
df["분기idx"]  = quarter_index(df["기준_년분기_코드"])
df["순증감"]   = df["개업_점포_수"] - df["폐업_점포_수"]
df["순감소_현재"] = (df["순증감"] < 0).astype(int)

df = df.sort_values(["상권_코드", "서비스_업종_코드", "분기idx"]).reset_index(drop=True)
g = df.groupby(["상권_코드", "서비스_업종_코드"], sort=False)
df["y"] = g["순감소_현재"].shift(-1)
df["다음분기idx"] = g["분기idx"].shift(-1)
df.loc[(df["다음분기idx"] - df["분기idx"]) != 1, "y"] = np.nan

print(f"라벨 생성 가능 {df['y'].notna().sum():,} / {len(df):,}")
print(f"무사건 칸(개업=0 & 폐업=0) {((df['개업_점포_수']==0)&(df['폐업_점포_수']==0)).mean():.1%}")
print(f"점포 {MIN_STORES}개 미만 {(df['전체_점포_수'] < MIN_STORES).mean():.1%}")

In [ ]:
steps = [("원본", df)]
d = df[df["y"].notna()].copy();                  steps.append(("라벨 존재", d))
d = d[d["전체_점포_수"] >= MIN_STORES];            steps.append((f"점포 {MIN_STORES}개 이상", d))
d = d[d["총_유동인구_수"].notna()];                steps.append(("유동인구 존재", d))
d = d[d["영역_면적"] > 0];                        steps.append(("면적 유효", d))

prev = None
for nm, s in steps:
    print(f"  {nm:16} {len(s):>8,}" + ("" if prev is None else f"  (-{prev-len(s):,})")); prev = len(s)
d["y"] = d["y"].astype(int)
print(f"\n최종 표본 {len(d):,} · 양성률 {d['y'].mean():.1%}")
display(d.groupby("기준_년분기_코드")["y"].agg(["size", "mean"]).round(3))

## 3-1. 로그 변환과 파생 변수

`영역_면적`이 최대/최소 1,000배 이상 차이나므로 로그 변환은 선택이 아니라 필수임.

In [ ]:
d["log_점포수"]   = np.log(d["전체_점포_수"])
d["log_유동인구"] = np.log1p(d["총_유동인구_수"])
d["log_면적"]     = np.log(d["영역_면적"])
d["log_집객시설"] = np.log1p(d["집객시설_수"].fillna(0))
d["점포밀도"]     = d["전체_점포_수"] / d["상권_전체점포"]
d["프랜차이즈비율"] = d["프랜차이즈_점포_수"] / d["전체_점포_수"]
d["점포당매출"]   = d["당월_매출_금액"] / d["전체_점포_수"]

display(d[["log_점포수", "log_유동인구", "log_면적", "log_집객시설",
           "점포밀도", "프랜차이즈비율", "직장인구_비중"]]
        .describe().loc[["mean", "std", "min", "max"]].round(3))
d.to_pickle(PROC / "03_panel.pkl")
print(f"[저장] 03_panel.pkl {d.shape}")

---
# STEP 4. 기초 EDA

데이터가 분석 가능한 상태인지 확인함.

In [ ]:
d = pd.read_pickle(PROC / "03_panel.pkl")
print(f"표본 {d.shape} · 양성률 {d['y'].mean():.1%}")
display(pd.crosstab(d["서비스_업종_코드_명"], d["상권유형"]))
print("※ 관광특구는 표본이 적어 별도 결론을 내지 않음.")

In [ ]:
t_type = d.groupby("상권유형")["y"].agg(칸수="size", 순감소율="mean").sort_values("순감소율")
gm = d[d["상권유형"] == TARGET_TYPE]
t_gu = (gm.groupby("자치구")["y"].agg(칸수="size", 순감소율="mean")
        .sort_values("순감소율", ascending=False))
display(t_type.round(3)); display(t_gu.round(3).head(10))

## 4-1. 상권유형별 차이 — 검정과 효과크기

표본이 수만 건이라 **p-value는 거의 항상 유의하게** 나옴.
실질적 차이인지는 **효과크기(Cramér's V)** 가 판정함. 둘을 반드시 함께 보고함.

In [ ]:
ct = pd.crosstab(d["상권유형"], d["y"])
chi2, p, dof, _ = chi2_contingency(ct, correction=False)
V = np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1)))
print(f"상권유형 × 순감소  chi2({dof}) = {chi2:.1f}, p = {p:.3g}, Cramér's V = {V:.3f}")
print("   V < 0.1 이면 통계적으로 유의해도 실질적 차이는 작다고 해석함\n")

groups = [g["y"].values for _, g in d.groupby("상권유형")]
H, pk = kruskal(*groups)
eps2 = (H - len(groups) + 1) / (len(d) - len(groups))
print(f"Kruskal–Wallis  H = {H:.1f}, p = {pk:.3g}, ε² = {eps2:.4f}")

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(14, 9))
ax[0,0].barh(t_type.index, t_type["순감소율"], color="#5b8db8")
ax[0,0].axvline(d["y"].mean(), ls="--", c="k", lw=1, label="전체 평균")
ax[0,0].set_title("상권유형별 순감소율", weight="bold"); ax[0,0].legend(fontsize=8)

ax[0,1].barh(t_gu.index, t_gu["순감소율"], color="#c0703a")
ax[0,1].axvline(gm["y"].mean(), ls="--", c="k", lw=1)
ax[0,1].set_title(f"자치구별 순감소율 ({TARGET_TYPE})", weight="bold"); ax[0,1].tick_params(labelsize=7)

ax[1,0].hist(d["전체_점포_수"], bins=80, color="#888", log=True)
ax[1,0].set_title("점포 수 원척도 (로그 y축) — 강한 우편포", weight="bold")

cov = ["log_점포수", "log_유동인구", "log_면적", "log_집객시설", "직장인구_비중"]
corr = d[cov].corr(method="spearman")
im = ax[1,1].imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax[1,1].set_xticks(range(len(cov))); ax[1,1].set_xticklabels(cov, rotation=30, fontsize=7)
ax[1,1].set_yticks(range(len(cov))); ax[1,1].set_yticklabels(cov, fontsize=7)
for i in range(len(cov)):
    for j in range(len(cov)):
        ax[1,1].text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center", fontsize=8)
ax[1,1].set_title("공변량 상관 (스피어만)", weight="bold")
plt.tight_layout(); plt.savefig(FIG / "step4_basic_eda.png", dpi=130); plt.show()
print("※ |r| > 0.8 인 쌍이 있으면 모형에서 하나를 빼는 것을 검토함")

---
# STEP 5. 기준선과 가설 EDA

모델을 만들기 전에 **아무것도 안 하는 규칙이 얼마나 맞히는지** 먼저 재 둠.
이 값을 넘지 못하면 모델을 쓸 이유가 없음.

## 5-1. 기준선 세 가지

| 기준선 | 규칙 |
|---|---|
| 무작위 | 전체 양성률(기저율) |
| 관성 | 직전 분기에 순감소였으면 위험으로 찍음 |
| 상권변화지표 | 서울시 공식 지표가 `HL 상권축소`이면 위험으로 찍음 |

세 기준선과 모델을 **같은 혼동행렬** 위에서 비교함.
`Lift = 정밀도 ÷ 기저율` 이며, 1.0이면 무작위와 같음.

In [ ]:
def evaluate(y_true, pred, name):
    y_true = np.asarray(y_true); pred = np.asarray(pred)
    tp = int(((pred == 1) & (y_true == 1)).sum())
    fp = int(((pred == 1) & (y_true == 0)).sum())
    fn = int(((pred == 0) & (y_true == 1)).sum())
    base = y_true.mean()
    prec = tp / (tp + fp) if tp + fp else np.nan
    rec  = tp / (tp + fn) if tp + fn else np.nan
    return {"기준선": name, "위험으로 찍은 수": tp + fp, "정밀도": prec, "재현율": rec,
            "Lift": prec / base if base else np.nan}

base_df = d[d["상권_변화_지표"].notna()].copy()
rows = [
    evaluate(base_df["y"], np.ones(len(base_df)),          "무작위(전부 위험)"),
    evaluate(base_df["y"], base_df["순감소_현재"],           "관성(직전 분기)"),
    evaluate(base_df["y"], base_df["기준선_위험"],           "상권변화지표(HL)"),
]
bl = pd.DataFrame(rows)
bl[["정밀도", "재현율", "Lift"]] = bl[["정밀도", "재현율", "Lift"]].round(3)
display(bl)
print(f"기저율 = {base_df['y'].mean():.1%}  ← 모델은 이 표의 Lift를 넘어야 의미가 있음")

In [ ]:
# 관성이 실제로 얼마나 강한지 — 2x2 분할표 + 카이제곱 + 효과크기
ct2 = pd.crosstab(base_df["순감소_현재"], base_df["y"])
chi2, p, dof, _ = chi2_contingency(ct2, correction=False)
V = np.sqrt(chi2 / ct2.values.sum())
OR = (ct2.iloc[1,1] * ct2.iloc[0,0]) / (ct2.iloc[1,0] * ct2.iloc[0,1])
display(ct2)
print(f"chi2({dof}) = {chi2:.1f}, p = {p:.3g}, Cramér's V = {V:.3f}, 오즈비 = {OR:.2f}")
print("→ p는 사실상 0이지만 V가 작으면 관성만으로는 예측력이 약하다는 뜻임. "
      "p-value만 보고 결론을 내면 안 되는 이유의 실물임.")

## 5-2. 업종별 순감소율과 규모 교란

처리군은 **데이터를 보고 고르지 않음.** 최댓값을 고른 뒤 그 격차를 검정하면 선택 효과가 들어감.
표본이 가장 크고 골목상권을 대표하는 **한식음식점**을 선험적으로 처리군으로 고정함.

In [ ]:
gm = d[d["상권유형"] == TARGET_TYPE].copy()
t_ind = (gm.groupby("서비스_업종_코드_명")
         .agg(순감소율=("y", "mean"), 칸수=("y", "size"),
              평균점포=("전체_점포_수", "mean"), 중앙유동인구=("총_유동인구_수", "median"))
         .sort_values("순감소율", ascending=False))
display(t_ind.round(3))

TREAT = "한식음식점"          # 선험적 고정 (표본 최대 · 대표 업종)
gm["처리"] = (gm["서비스_업종_코드_명"] == TREAT).astype(int)
p1 = gm.loc[gm["처리"] == 1, "y"].mean(); p0 = gm.loc[gm["처리"] == 0, "y"].mean()
print(f"\n처리군 {TREAT} n={int(gm['처리'].sum()):,} 순감소율 {p1:.1%}")
print(f"대조군 기타 외식  n={int((1-gm['처리']).sum()):,} 순감소율 {p0:.1%}")
print(f"단순 격차 {p1-p0:+.1%}p")

In [ ]:
r = np.corrcoef(t_ind["평균점포"], t_ind["순감소율"])[0, 1]
print(f"업종 평균점포수 ↔ 순감소율 상관 r = {r:.3f}")
print("r이 높으면 '업종 고유 위험'이 아니라 '규모 차이'일 가능성이 큼\n")

gm["규모구간"] = pd.qcut(gm["log_점포수"], 5, labels=[f"Q{i}" for i in range(1, 6)])
piv = gm.pivot_table(index="규모구간", columns="처리", values="y",
                     aggfunc=["mean", "size"], observed=True)
tbl = pd.DataFrame({"대조군_순감소율": piv[("mean", 0)], "처리군_순감소율": piv[("mean", 1)],
                    "대조n": piv[("size", 0)].astype("Int64"), "처리n": piv[("size", 1)].astype("Int64")})
tbl["격차"] = tbl["처리군_순감소율"] - tbl["대조군_순감소율"]
display(tbl.round(3))
print("※ 규모 구간을 고정했을 때 격차가 줄거나 부호가 뒤집히면, 단순 격차는 규모 교란의 산물임")

## 5-3. 공변량 균형 (SMD)

|SMD| < 0.1 균형 / 0.1~0.25 불균형 / 0.25 초과 심각.
매칭 **후** 모든 공변량이 0.1 미만이 되어야 "조건을 맞췄다"고 말할 수 있음.

In [ ]:
COVARIATES = ["log_점포수", "log_유동인구", "log_면적", "log_집객시설", "직장인구_비중"]
def smd(a, b):
    return (a.mean() - b.mean()) / np.sqrt((a.var() + b.var()) / 2)

rows = []
for c in COVARIATES:
    a = gm.loc[gm["처리"] == 1, c].dropna(); b = gm.loc[gm["처리"] == 0, c].dropna()
    s = smd(a, b)
    rows.append({"공변량": c, "처리군평균": a.mean(), "대조군평균": b.mean(), "SMD": s,
                 "판정": "균형" if abs(s) < .1 else ("불균형" if abs(s) < .25 else "심각한 불균형")})
display(pd.DataFrame(rows).round(3))

for c in ["log_점포수", "log_유동인구"]:
    a = gm.loc[gm["처리"] == 1, c]; b = gm.loc[gm["처리"] == 0, c]
    lo, hi = max(a.min(), b.min()), min(a.max(), b.max())
    print(f"{c:12} 공통지지영역 [{lo:.2f}, {hi:.2f}] → 표본의 {((gm[c]>=lo)&(gm[c]<=hi)).mean():.1%} 포함")
gm.to_pickle(PROC / "05_golmok.pkl")
print(f"\n[저장] 05_golmok.pkl {gm.shape}")

---
## 정리와 다음 단계

| 단계 | 산출물 | 확인한 것 |
|---|---|---|
| STEP 1 | 원본 7종 적재 | 컬럼 대응 검증 · 집객시설 0 처리 · 기간 확정 |
| STEP 2 | `02_merged.pkl` | 좌결합 6회 모두 행 수 유지 · 결측률 기록 |
| STEP 3 | `03_panel.pkl` | 라벨 `y(t+1)` · 표본 필터 단계별 손실 |
| STEP 4 | 기초 EDA | 유형·자치구 차이를 **p와 효과크기 함께** 보고 |
| STEP 5 | `05_golmok.pkl` | **기준선 3종** · 규모 교란 진단 · SMD |

**다음 노트북(예측 모델링)에서 할 일**

1. 시점 분할 — 학습 `2023Q1~2025Q2` / 검증 `2025Q3~Q4`, 셔플하지 않음
2. `t+1` 시점 정보를 어떤 형태로도 피처에 넣지 않음
3. 로지스틱 회귀를 기준으로 랜덤포레스트·부스팅과 비교, 성능 차가 작으면 해석 쉬운 쪽을 채택함
4. 불균형이므로 정확도 대신 **PR-AUC와 리프트**로 평가하고, **STEP 5의 기준선 3종을 넘는지** 확인함
5. SHAP으로 «무엇이 위험을 만드는가»를 변수 단위로 분해함

**이 노트북에서 다루지 않은 것** — 임대료(자치구 단위만 존재), 온라인 관심도(외부 API),
공간 분석(좌표는 적재해 두었으나 GIS는 별도 노트북).